### Import packages

In [6]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from sklearn.feature_extraction.text import TfidfTransformer


### Import core tables

In [7]:
CORPUS = pd.read_csv("CORPUS.csv", sep="|")
LIB = pd.read_csv("LIB.csv", sep="|")
VOCAB = pd.read_csv("VOCAB.csv", sep="|")

### BOW Table

In [8]:
# Bag-of-words table
BOW = CORPUS.groupby(["doc_id", "term_str"]).agg(
    n=("term_str", "count")
).reset_index()

# Document frequency
df = BOW.groupby("term_str")["doc_id"].nunique()

# Total number of documents
D = CORPUS["doc_id"].nunique()

# Add tfidf
BOW["tfidf"] = BOW.apply(
    lambda row: row["n"] * np.log2(D / df[row["term_str"]]),
    axis=1
)

# Remove stopwords and remove junk tokens
BOW = BOW[BOW["term_str"].isin(
    VOCAB.loc[VOCAB["stop"] == False, "term_str"]
)]

BOW = BOW[BOW["term_str"].str.len() > 2]

BOW.to_csv("BOW.csv", sep="|", index=False)

BOW.head()

,doc_id,term_str,n,tfidf
4,0,bed,1,7.957828
5,0,choice,1,6.034995
7,0,dumb,1,9.372865
10,0,get,1,2.506616
12,0,going,1,4.429048


In [9]:
print(f"Number of observations in BOW: {len(BOW)}")
print(f"List of columns in BOW: {BOW.columns.tolist()}")

Number of observations in BOW: 326807
List of columns in BOW: ['doc_id', 'term_str', 'n', 'tfidf']


### DTM Table

In [ ]:
# make sure doc_id and term_str are categorical so we get stable row/column indices
BOW["doc_id"] = BOW["doc_id"].astype("category")
BOW["term_str"] = BOW["term_str"].astype("category")

# sparse count matrix
DTM = csr_matrix(
    (
        BOW["n"],
        (BOW["doc_id"].cat.codes, BOW["term_str"].cat.codes)
    )
)

# Labels for rows/columns
dtm_doc_ids = BOW["doc_id"].cat.categories
dtm_terms = BOW["term_str"].cat.categories

save_npz("DTM.npz", DTM)
pd.Series(dtm_doc_ids).to_csv("DTM_doc_ids.csv", sep="|", index=False)
pd.Series(dtm_terms).to_csv("DTM_terms.csv", sep="|", index=False)

### TFIDF

In [11]:
tfidf_transformer = TfidfTransformer(norm=None, use_idf=True, smooth_idf=False, sublinear_tf=False)
TFIDF = tfidf_transformer.fit_transform(DTM)

save_npz("TFIDF.npz", TFIDF)
pd.Series(dtm_doc_ids).to_csv("TFIDF_doc_ids.csv", sep="|", index=False)
pd.Series(dtm_terms).to_csv("TFIDF_terms.csv", sep="|", index=False)

### Reduced and Normalized TFIDF_L2

In [12]:
# Choose significant words 
term_df = BOW.groupby("term_str")["doc_id"].nunique()
significant_terms = term_df[term_df >= 5].index 

BOW_sig = BOW[BOW["term_str"].isin(significant_terms)].copy()

BOW_sig["doc_id"] = BOW_sig["doc_id"].astype("category")
BOW_sig["term_str"] = BOW_sig["term_str"].astype("category")

DTM_sig = csr_matrix(
    (BOW_sig["n"], (BOW_sig["doc_id"].cat.codes, BOW_sig["term_str"].cat.codes))
)

sig_doc_ids = BOW_sig["doc_id"].cat.categories
sig_terms = BOW_sig["term_str"].cat.categories

tfidf_l2 = TfidfTransformer(
    norm="l2",
    use_idf=True,
    smooth_idf=True,
    sublinear_tf=False
).fit_transform(DTM_sig)

save_npz("TFIDF_L2.npz", tfidf_l2)
pd.Series(sig_doc_ids).to_csv("TFIDF_L2_doc_ids.csv", sep="|", index=False)
pd.Series(sig_terms).to_csv("TFIDF_L2_terms.csv", sep="|", index=False)

C:\Users\linod\AppData\Local\Temp\ipykernel_19420\1425617891.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  term_df = BOW.groupby("term_str")["doc_id"].nunique()


In [13]:
print(f"Number of significant words in TFIDF_L2: {len(sig_terms)}")

Number of significant words in TFIDF_L2: 25262
